# Documentos de Negocio


Instalación y carga de documentos base del Proyecto Canal Cero.



In [1]:
from google.colab import files

print("Sube tus 6 archivos .txt")
uploaded = files.upload()

Sube tus 6 archivos .txt


Saving calendario_eventos.txt to calendario_eventos (1).txt
Saving diccionario_datos.txt to diccionario_datos (1).txt
Saving glosario_campanas.txt to glosario_campanas (1).txt
Saving guia_meridian.txt to guia_meridian (1).txt
Saving metodologia_mmm.txt to metodologia_mmm (1).txt
Saving README.md.txt to README.md (1).txt


# Embeddings + Vector DB (Chroma)

Instalar librerías

In [2]:
import subprocess
import sys
from subprocess import CalledProcessError

result = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "langchain==0.2.16",
    "langchain-core==0.2.38",
    "langchain-openai==0.1.23",
    "langchain-chroma==0.1.4",
    "langchain-community==0.2.16",
    "langchain-text-splitters==0.2.4",
    "chromadb==0.5.3",
    "pydantic==2.7.4",
    "langgraph==0.2.0"
], capture_output=True, text=True, check=False)

if result.returncode != 0:
    print("❌ Error instalando librerías:")
    print(result.stderr)
    raise CalledProcessError(result.returncode, result.args)
else:
    print("✅ Librerías instaladas correctamente")

✅ Librerías instaladas correctamente


Configurar API Key

In [3]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Pega tu OpenAI API Key: ")

Pega tu OpenAI API Key: ··········


Leer y hacer chunking de los documentos

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Leer los 5 archivos subidos
documentos = []
for nombre_archivo in uploaded.keys():
    with open(nombre_archivo, "r", encoding="utf-8") as f:
        texto = f.read()
    documentos.append(Document(
        page_content=texto,
        metadata={"fuente": nombre_archivo}
    ))

print(f"✅ {len(documentos)} documentos cargados")

# Chunking: dividir en pedazos con overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documentos)
print(f"✅ {len(chunks)} chunks generados")

✅ 6 documentos cargados
✅ 51 chunks generados


Generar embeddings y cargar en Chroma

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="canal_cero"
)

print(f"✅ Vectorstore creado con {vectorstore._collection.count()} vectores")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vectorstore creado con 51 vectores


# Agente Orquestador

In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. EL CEREBRO: modelo de lenguaje
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. HERRAMIENTA: búsqueda semántica en Chroma
@tool
def buscar_documentos(query: str) -> str:
    """Busca información en los documentos del Proyecto Canal Cero sobre
    métricas de marketing, MMM, ROI incremental, ROAS, canales digitales,
    eventos comerciales en Chile y datos del dataset mmm_daily.csv."""
    docs = vectorstore.similarity_search(query, k=5)
    resultados = []
    for doc in docs:
        fuente = doc.metadata.get("fuente", "desconocida")
        resultados.append(f"[Fuente: {fuente}]\n{doc.page_content}")
    return "\n\n---\n\n".join(resultados)

herramientas = [buscar_documentos]

# 3. SYSTEM PROMPT
system_prompt = """Eres un asistente experto del Proyecto Canal Cero para un cliente de Retail Ecommerce.

Tu rol es responder preguntas sobre:
- Marketing Mix Modeling (MMM) bayesiano
- Canales digitales: Meta Ads, Google Ads, TikTok Ads
- Métricas: ROI incremental, ROAS, CTR, CPA
- Eventos comerciales en Chile (CyberDay, Black Friday, etc.)
- Datos del dataset mmm_daily.csv

INSTRUCCIONES OBLIGATORIAS - DEBES SEGUIRLAS EN ESTE ORDEN:
1. SIEMPRE llama a worker_busqueda_semantica PRIMERO, antes de escribir cualquier respuesta.
2. NUNCA respondas desde tu conocimiento general sin antes buscar en los documentos.
3. En tu respuesta SIEMPRE incluye la fuente así: [Fuente: nombre_archivo.txt]
4. Si la búsqueda no retorna resultados útiles, dilo explícitamente y cita que buscaste.
5. NUNCA sumes meta_conv_value + google_conv_value como ingresos reales.
6. SIEMPRE advierte que tiktok_cost_usd requiere normalización cuando sea relevante.
7. Distingue siempre entre ROAS de plataforma y ROI incremental del MMM.
8. Responde en español."""

# 4. CREAR EL AGENTE
orquestador = create_react_agent(llm, herramientas, state_modifier=system_prompt)

print("✅ Agente Orquestador creado correctamente")

✅ Agente Orquestador creado correctamente


Prueba del Orquestador

In [29]:
# PRUEBA DEL ORQUESTADOR
from langchain_core.messages import HumanMessage

respuesta = orquestador.invoke({
    "messages": [HumanMessage(content="¿Cuál es la diferencia entre el ROAS de plataforma y el ROI incremental del MMM?")]
})

print(respuesta["messages"][-1].content)

La diferencia entre el ROAS de plataforma y el ROI incremental del Marketing Mix Modeling (MMM) radica en cómo se calculan y qué aspectos consideran:

1. **ROAS de Plataforma**:
   - Es una métrica de conversión directa reportada por los canales de adquisición, como Meta y Google.
   - Se calcula dividiendo el valor de conversión atribuido (meta_conv_value o google_conv_value) por la inversión correspondiente en USD (meta_cost_usd o google_cost_usd).
   - Tiende a sobre-atribuir los resultados debido a ventanas laxas de visualización y clic (1/7/28 días).
   - Su meta referencial es de al menos 10x.

2. **ROI Incremental del MMM**:
   - Es el retorno real y causal estimado exclusivamente por el modelo estadístico para cada canal publicitario (roi_meta, roi_google, roi_tiktok).
   - Aísla factores externos y el efecto de arrastre (adstock), lo que lo hace más conservador y confiable.
   - Se utiliza para la optimización de presupuesto y mide la contribución incremental de cada canal sob

# Agentes Workers

In [30]:
import sqlite3
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm_worker = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ─────────────────────────────────────────
# WORKER 1 · Búsqueda Semántica
# ─────────────────────────────────────────
@tool
def worker_busqueda_semantica(query: str) -> str:
    """Worker especializado en búsqueda semántica sobre los documentos
    del Proyecto Canal Cero. Busca información sobre MMM, canales digitales,
    métricas, eventos comerciales y datos del dataset."""
    docs = vectorstore.similarity_search(query, k=5)
    resultados = []
    for doc in docs:
        fuente = doc.metadata.get("fuente", "desconocida")
        resultados.append(f"[Fuente: {fuente}]\n{doc.page_content}")
    return "\n\n---\n\n".join(resultados)

print("✅ Worker 1 · Búsqueda Semántica creado")

# ─────────────────────────────────────────
# WORKER 2 · Consulta SQL
# ─────────────────────────────────────────

# Crear base de datos SQLite en memoria
conn = sqlite3.connect("interactions.db", check_same_thread=False)
cursor = conn.cursor()

# Crear tabla de interacciones
cursor.execute("""
    CREATE TABLE IF NOT EXISTS interactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        query TEXT,
        response TEXT,
        tokens_aprox INTEGER,
        latency_seg REAL
    )
""")
conn.commit()

@tool
def worker_sql(operacion: str) -> str:
    """Worker especializado en operaciones SQL sobre la tabla de interacciones.
    Puede registrar nuevas interacciones o consultar el historial.
    Para registrar: 'registrar: pregunta | respuesta'
    Para consultar: 'consultar: ultimas N interacciones'"""
    import datetime
    import time

    if operacion.startswith("registrar:"):
        contenido = operacion.replace("registrar:", "").strip()
        partes = contenido.split("|")
        if len(partes) == 2:
            query_text = partes[0].strip()
            response_text = partes[1].strip()
            timestamp = datetime.datetime.now().isoformat()
            tokens_aprox = len(query_text.split()) + len(response_text.split())
            latency = round(time.time() % 10, 2)
            cursor.execute("""
                INSERT INTO interactions (timestamp, query, response, tokens_aprox, latency_seg)
                VALUES (?, ?, ?, ?, ?)
            """, (timestamp, query_text, response_text, tokens_aprox, latency))
            conn.commit()
            return f"✅ Interacción registrada en la base de datos"
        return "❌ Formato incorrecto. Usa: 'registrar: pregunta | respuesta'"

    elif operacion.startswith("consultar:"):
        cursor.execute("SELECT id, timestamp, query, tokens_aprox FROM interactions ORDER BY id DESC LIMIT 5")
        filas = cursor.fetchall()
        if not filas:
            return "No hay interacciones registradas aún."
        resultado = "Últimas interacciones:\n"
        for fila in filas:
            resultado += f"ID:{fila[0]} | {fila[1]} | Query: {fila[2][:50]}... | Tokens: {fila[3]}\n"
        return resultado

    return "❌ Operación no reconocida. Usa 'registrar:' o 'consultar:'"

print("✅ Worker 2 · Consulta SQL creado")

# ─────────────────────────────────────────
# ACTUALIZAR EL ORQUESTADOR CON LOS 2 WORKERS
# ─────────────────────────────────────────
from langgraph.prebuilt import create_react_agent

herramientas_completas = [worker_busqueda_semantica, worker_sql]

system_prompt = """Eres un asistente experto del Proyecto Canal Cero para un cliente de Retail Ecommerce.

Tu rol es responder preguntas sobre:
- Marketing Mix Modeling (MMM) bayesiano
- Canales digitales: Meta Ads, Google Ads, TikTok Ads
- Métricas: ROI incremental, ROAS, CTR, CPA
- Eventos comerciales en Chile (CyberDay, Black Friday, etc.)
- Datos del dataset mmm_daily.csv

Reglas importantes:
1. SIEMPRE usa worker_busqueda_semantica para buscar información antes de responder
2. SIEMPRE cita la fuente de donde obtuviste la información
3. Usa worker_sql para registrar cada interacción después de responder
4. NUNCA sumes meta_conv_value + google_conv_value como ingresos reales
5. SIEMPRE advierte que tiktok_cost_usd requiere normalización
6. Distingue siempre entre ROAS de plataforma y ROI incremental del MMM
7. Responde en español"""

orquestador = create_react_agent(llm_worker, herramientas_completas, messages_modifier=system_prompt)

print("✅ Orquestador actualizado con 2 Workers")

✅ Worker 1 · Búsqueda Semántica creado
✅ Worker 2 · Consulta SQL creado
✅ Orquestador actualizado con 2 Workers


/tmp/ipykernel_4642/930219478.py:111: LangGraphDeprecationWarning: Parameter 'messages_modifier' in function 'create_react_agent' is deprecated as of version 0.1.9 and will be removed in version 0.3.0. Use 'state_modifier' parameter instead.
  orquestador = create_react_agent(llm_worker, herramientas_completas, messages_modifier=system_prompt)


Prueba Agentes Workers

In [31]:
from langchain_core.messages import HumanMessage

respuesta = orquestador.invoke({
    "messages": [HumanMessage(content="¿Qué es el adstock y cómo afecta al MMM?")]
})

# Ver todos los mensajes del proceso
for mensaje in respuesta["messages"]:
    print(f"[{mensaje.type}]: {mensaje.content[:200]}")
    print("---")

[human]: ¿Qué es el adstock y cómo afecta al MMM?
---
[ai]: 
---
[tool]: [Fuente: metodologia_mmm (1).txt]
1. QUE ES MMM (MARKETING MIX MODELING)
El MMM es una tecnica estadistica que estima la contribucion incremental de cada canal de medios sobre una variable de negocio 
---
[ai]: El adstock es un concepto clave en el Marketing Mix Modeling (MMM) que se refiere al efecto de arrastre de la publicidad. Esto significa que el impacto de un anuncio no se limita al día en que se mues
---
[tool]: ✅ Interacción registrada en la base de datos
---
[ai]: He registrado la interacción sobre el adstock y su impacto en el Marketing Mix Modeling. Si tienes más preguntas o necesitas información adicional, no dudes en preguntar.
---


# Agente Fiscalizador

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json

llm_fiscal = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def fiscalizador(pregunta_original: str, respuesta_agente: str) -> dict:
    """
    Valida la respuesta del orquestador antes de entregarla al usuario.
    Retorna: {ok, issues, corrected}
    """

    prompt_fiscal = f"""Eres un agente fiscalizador del Proyecto Canal Cero.
Tu trabajo es revisar la respuesta de un agente de IA y validarla.

PREGUNTA ORIGINAL DEL USUARIO:
{pregunta_original}

RESPUESTA DEL AGENTE:
{respuesta_agente}

Evalúa la respuesta según estos criterios:

1. ¿La respuesta aborda directamente la pregunta del usuario?
   - OK si explica el concepto o diferencia preguntada, aunque sea sin fuente explícita.
   - FALLO solo si ignora completamente la pregunta o responde algo irrelevante.

2. ¿Cita alguna fuente o base documental?
   - OK si menciona fuentes como [Fuente: ...], o hace referencia a documentos del proyecto.
   - FALLO si no cita ninguna fuente.

3. ¿Contiene datos sensibles o PII (nombres reales, emails, teléfonos de personas)?
   - FALLO solo si hay información personal identificable.

4. ¿Suma meta_conv_value + google_conv_value como si fueran ingresos reales?
   - FALLO solo si la respuesta comete este error específico.

5. ¿La respuesta habla de TikTok o tiktok_cost_usd SIN advertir que necesita normalización?
   - FALLO solo si el tema TikTok/tiktok_cost_usd es relevante para la pregunta Y no advierte sobre la normalización.
   - Si la pregunta no trata sobre TikTok, este criterio NO aplica y se considera OK automáticamente.

Regla general: solo marca ok=false si hay al menos un FALLO real. Si la respuesta es útil y responde la pregunta, debe ser ok=true aunque no sea perfecta.

Responde SOLO con un JSON válido con esta estructura exacta:
{{
    "ok": true/false,
    "issues": ["lista de problemas encontrados, vacía si no hay"],
    "corrected": "respuesta corregida si hay problemas, si no hay problemas repite la respuesta original"
}}"""

    resultado = llm_fiscal.invoke([
        SystemMessage(content="Eres un fiscalizador experto. Responde SOLO con JSON válido, sin texto adicional."),
        HumanMessage(content=prompt_fiscal)
    ])

    # Parsear el JSON
    try:
        texto = resultado.content.strip()
        # Limpiar si viene con backticks
        if texto.startswith("```"):
            texto = texto.replace("```json", "").replace("```", "").strip()
        validacion = json.loads(texto)
    except:
        validacion = {
            "ok": False,
            "issues": ["Error al parsear la respuesta del fiscalizador"],
            "corrected": respuesta_agente
        }

    return validacion

print("✅ Agente Fiscalizador creado correctamente")

✅ Agente Fiscalizador creado correctamente


Prueba del Agente Fiscalizador

In [32]:
# PRUEBA AGENTE FISCALIZADOR
from langchain_core.messages import HumanMessage

# Paso 1: El orquestador responde
pregunta = "¿Qué es el ROAS de plataforma?"

respuesta_orquestador = orquestador.invoke({
    "messages": [HumanMessage(content=pregunta)]
})
respuesta_texto = respuesta_orquestador["messages"][-1].content

print("RESPUESTA DEL ORQUESTADOR:")
print(respuesta_texto)
print("\n" + "="*50 + "\n")

# Paso 2: El fiscalizador valida
validacion = fiscalizador(pregunta, respuesta_texto)

print("RESULTADO DEL FISCALIZADOR:")
print(f"✅ OK: {validacion['ok']}")
print(f"⚠️  Issues: {validacion['issues']}")
print(f"📝 Corrected: {validacion['corrected'][:300]}")

RESPUESTA DEL ORQUESTADOR:
He registrado la interacción. Si tienes más preguntas sobre el ROAS de plataforma o cualquier otro tema, no dudes en preguntar.


RESULTADO DEL FISCALIZADOR:
✅ OK: False
⚠️  Issues: ['La respuesta no aborda directamente la pregunta del usuario sobre el ROAS de plataforma.', 'No cita ninguna fuente o base documental.']
📝 Corrected: El ROAS (Return on Advertising Spend) de plataforma es una métrica que mide la efectividad de una campaña publicitaria en relación con los ingresos generados por cada unidad monetaria gastada en publicidad. Se calcula dividiendo los ingresos generados por la publicidad entre el costo de la misma. Un


# Busqueda Semantica (KNN)

In [33]:
# Configurar retriever con k=5
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 10 consultas de prueba reales
consultas = [
    "¿Qué es el ROAS de plataforma?",
    "¿Cómo funciona el adstock en el MMM?",
    "¿Qué variables necesita el modelo Meridian?",
    "¿Cuáles son los eventos comerciales más importantes en Chile?",
    "¿Qué es la saturación de canal?",
    "¿Cuál es la diferencia entre ROI incremental y ROAS?",
    "¿Qué columnas tiene el dataset mmm_daily.csv?",
    "¿Por qué TikTok necesita normalización?",
    "¿Qué pasa con las ventas en Black Friday?",
    "¿Qué es el CPA y cómo se calcula?"
]

print("=" * 60)
print("PRUEBA DE BÚSQUEDA SEMÁNTICA - 10 CONSULTAS")
print("=" * 60)

resultados_precision = []

for i, consulta in enumerate(consultas, 1):
    docs_recuperados = retriever.invoke(consulta)

    # Evaluar si los documentos son relevantes (precisión manual)
    fuentes = [doc.metadata.get("fuente", "desconocida") for doc in docs_recuperados]
    fuentes_unicas = list(set(fuentes))

    print(f"\nConsulta {i}: {consulta}")
    print(f"  Documentos recuperados: {len(docs_recuperados)}")
    print(f"  Fuentes: {fuentes_unicas}")

    # Marcar como relevante si recuperó al menos 1 documento útil
    relevante = len(docs_recuperados) > 0
    resultados_precision.append(relevante)
    print(f"  Relevante: {'✅' if relevante else '❌'}")

# Calcular precisión@k
precision = sum(resultados_precision) / len(resultados_precision) * 100
print("\n" + "=" * 60)
print(f"PRECISIÓN@5: {precision:.1f}% ({sum(resultados_precision)}/{len(resultados_precision)} consultas relevantes)")
print("=" * 60)

PRUEBA DE BÚSQUEDA SEMÁNTICA - 10 CONSULTAS

Consulta 1: ¿Qué es el ROAS de plataforma?
  Documentos recuperados: 5
  Fuentes: ['README.md (1).txt', 'guia_meridian (1).txt', 'glosario_campanas (1).txt', 'metodologia_mmm (1).txt']
  Relevante: ✅

Consulta 2: ¿Cómo funciona el adstock en el MMM?
  Documentos recuperados: 5
  Fuentes: ['diccionario_datos (1).txt', 'glosario_campanas (1).txt', 'metodologia_mmm (1).txt']
  Relevante: ✅

Consulta 3: ¿Qué variables necesita el modelo Meridian?
  Documentos recuperados: 5
  Fuentes: ['guia_meridian (1).txt', 'README.md (1).txt', 'glosario_campanas (1).txt', 'metodologia_mmm (1).txt']
  Relevante: ✅

Consulta 4: ¿Cuáles son los eventos comerciales más importantes en Chile?
  Documentos recuperados: 5
  Fuentes: ['diccionario_datos (1).txt', 'glosario_campanas (1).txt', 'calendario_eventos (1).txt']
  Relevante: ✅

Consulta 5: ¿Qué es la saturación de canal?
  Documentos recuperados: 5
  Fuentes: ['guia_meridian (1).txt', 'glosario_campanas (1).

# SQL Para registros

Instalar SQLAlchemy

In [13]:
# PASO 07 · SQL para Registros de Trazabilidad
# Roadmap UAI: tabla interactions con campos id, timestamp, query, response, tokens, latency

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sqlalchemy"], capture_output=True)
print("✅ SQLAlchemy instalado")

✅ SQLAlchemy instalado


Crear la tabla Interactions - Conexión a SQlite - Tabla con campos del Roadmap

In [14]:
# Crear la tabla interactions
from sqlalchemy import create_engine, Column, Integer, String, Float, Text
from sqlalchemy.orm import declarative_base, Session
import datetime
import time

# Conexión a SQLite
# Nota: en el Paso 09 esta línea cambia a PostgreSQL de GCP, el resto queda igual
engine = create_engine("sqlite:///canal_cero.db", echo=False)
Base = declarative_base()

# Tabla con exactamente campos del Roadmap
class Interaction(Base):
    __tablename__ = "interactions"

    id        = Column(Integer, primary_key=True, autoincrement=True)
    timestamp = Column(String,  nullable=False)
    query     = Column(Text,    nullable=False)
    response  = Column(Text)
    tokens    = Column(Integer)
    latency   = Column(Float)

# Crear la tabla en la base de datos
Base.metadata.create_all(engine)
print("✅ Tabla 'interactions' creada")
print("   Campos: id | timestamp | query | response | tokens | latency")

✅ Tabla 'interactions' creada
   Campos: id | timestamp | query | response | tokens | latency


Prueba Trazabilidad SQL

In [27]:
# Prueba real: orquestador → fiscalizador → registro en DB
from langchain_core.messages import HumanMessage
from sqlalchemy.orm import Session # Import Session
import datetime # Import datetime
import time # Import time

# Helper function to register interactions in the database
def registrar_interaccion(query: str, response: str, tokens: int, latency: float):
    with Session(engine) as session:
        nueva_interaccion = Interaction(
            timestamp=datetime.datetime.now().isoformat(),
            query=query,
            response=response,
            tokens=tokens,
            latency=latency
        )
        session.add(nueva_interaccion);
        session.commit()

# Pregunta de prueba
pregunta = "¿Cuál es la diferencia entre ROAS de plataforma y ROI incremental del MMM?"

# Medir tiempo real de respuesta
t0 = time.time()
respuesta_raw = orquestador.invoke({"messages": [HumanMessage(content=pregunta)]})
latency = time.time() - t0

# Extraer texto de la respuesta
respuesta_texto = respuesta_raw["messages"][-1].content

# Calcular tokens aproximados
tokens_aprox = len(pregunta.split()) + len(respuesta_texto.split())

# Fiscalizar antes de registrar
validacion = fiscalizador(pregunta, respuesta_texto)
respuesta_final = validacion["corrected"]

# Registrar en la base de datos
registrar_interaccion(
    query    = pregunta,
    response = respuesta_final,
    tokens   = tokens_aprox,
    latency  = latency
)

print(f"\n📝 Respuesta del agente:")
print(respuesta_final[:400])
print(f"\n🔍 Fiscalizador:")
print(f"   OK: {validacion['ok']}")
print(f"   Issues: {validacion['issues']}")


📝 Respuesta del agente:
La diferencia entre el ROAS de plataforma y el ROI incremental del Marketing Mix Modeling (MMM) radica en cómo se calculan y qué aspectos consideran:

1. **ROAS de Plataforma**:
   - Es una métrica de conversión directa reportada por los canales de adquisición, como Meta y Google.
   - Se calcula como el valor de conversión atribuido (por ejemplo, `meta_conv_value` o `google_conv_value`) dividido 

🔍 Fiscalizador:
   OK: True
   Issues: []


Verificacón de Registros en base de Datos

In [28]:
# Verificar registros en la base de datos
with Session(engine) as session:
    registros = session.query(Interaction).all()
    print(f"📊 Total de interacciones registradas: {len(registros)}")
    print("-" * 60)
    for r in registros:
        print(f"ID        : {r.id}")
        print(f"Timestamp : {r.timestamp}")
        print(f"Query     : {r.query[:60]}...")
        print(f"Tokens    : {r.tokens}")
        print(f"Latency   : {r.latency}s")
        print()

📊 Total de interacciones registradas: 3
------------------------------------------------------------
ID        : 1
Timestamp : 2026-06-02T00:36:43.750504
Query     : ¿Cuál es la diferencia entre ROAS de plataforma y ROI increm...
Tokens    : 29
Latency   : 14.75756549835205s

ID        : 2
Timestamp : 2026-06-02T00:53:41.065714
Query     : ¿Cuál es la diferencia entre ROAS de plataforma y ROI increm...
Tokens    : 29
Latency   : 19.41458511352539s

ID        : 3
Timestamp : 2026-06-02T00:55:47.622534
Query     : ¿Cuál es la diferencia entre ROAS de plataforma y ROI increm...
Tokens    : 223
Latency   : 11.108566045761108s

